## Stage 1 — Load and assemble raw job vacancy data

**Role:** Entry point of the pipeline. Reads all raw daily JSON snapshots from `data/data-pipeline/input/`, deduplicates records, cleans text, detects languages, and builds the processed dataset used by all downstream stages.

**Position:** This is the only notebook in Stage 1 required for the demo pipeline. Run it before any Stage 2 notebooks.

**Must be run manually** after placing input JSON files in `data/data-pipeline/input/`.

### Initialisation

Empty cell reserved for kernel warm-up.

## 1.1. Description

The first stage of the pipeline focuses on reading raw JSON files containing job advertisement data, cleaning up their textual content, and filtering new, unique records based on advertisement IDs. This step prepares the dataset for downstream processing tasks such as skill extraction and language classification.

## 1.2. Packages and configuration

Cleaning a memory of Jupyter Notebook before run:

Load the autoreload extension so that changes to `general.py` and `stage1.py` are picked up automatically without restarting the kernel. Run the shared bootstrap so the repository code can be imported without a hardcoded path. Call `g.clean_memory()` to release any variables left from a previous run.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import pandas as pd
import os
import general as g
g.clean_memory()
import stage1 as st1

Checking working directory:

Verify the working directory selected by `pipeline_bootstrap.py`. The bootstrap uses `notebooks/data-pipeline/` so paths from `.env` resolve consistently.

In [ ]:
%pwd

## 1.2. Create list of processing files

Build or update the Stage 1 process tracker. Scans `data/data-pipeline/input/` for JSON files and creates one row per file with columns tracking clean status and output paths. If the tracker pickle already exists, only newly added files are appended. The resulting DataFrame is displayed to confirm which files are registered.

In [ ]:
process_stage_df = st1.update_stage1_process_data(g.Config.INPUT_PATH, g.Config.STAGE1_PROCESS_PATH)
process_stage_df

Retrieve the list of files that have not yet been processed (i.e. `clean_status` is NaN). This list drives the processing loop below.

In [ ]:
files = st1.get_not_processed_files(process_stage_df)
files

Sanity check: if `True`, all files have already been processed and the loop below will be skipped.

In [ ]:
files[0] is None

Local override of the `clean_data_duplicates` function. Groups records by title, description, salary, and region to collapse duplicates, keeping the first ID and dates while averaging click counts across duplicate listings.

In [ ]:
def clean_data_duplicates(dtx):

    aggregated_by_few_cols = dtx.groupby(['title', 'description', 'min_salary', 'max_salary', 'region'], dropna=False).agg(
        {
            'id': 'first',
            'date_created': 'first',
            'date_expired': 'first',
            'salary_rate': 'first',
            'currency': 'first',
            'number_of_clicks': 'mean'
        }).reset_index()

    aggregated_by_few_cols = aggregated_by_few_cols[dtx.columns]

    return aggregated_by_few_cols


Ensure that the Stage 1 output folders exist before writing any files. Creates `output/` and `id_region_click/` if they are missing.

In [ ]:
g.check_folder_exists(g.Config.STAGE1_OUTPUT_PATH)
g.check_folder_exists(g.Config.STAGE1_ID_REGION_NCLICK_PATH)

Main processing loop. Iterates over all unprocessed input files and runs the full Stage 1 pipeline for each:

1. Reads the raw JSON and normalises dates
2. Deduplicates records by key fields
3. Saves the id/region/clicks snapshot
4. Filters to only new job IDs not seen in previous files
5. Cleans text (removes dates, salaries, emoji, normalises characters)
6. Detects and filters by language (keeps: en, uk, ru, cs, pl)
7. Updates the global unique ID database
8. Saves the cleaned output pickle and marks the file as complete in the tracker

In [ ]:
if files[0] is not None:
    for f_name in files:
        process_df = st1.update_stage1_process_data(g.Config.INPUT_PATH, g.Config.STAGE1_PROCESS_PATH)
        st1.preprocess_file(process_df, f_name, g.Config)
        print(f"Completed: {f_name}")

## Clean regional data

Reload the completed process tracker from disk. At this point all files should have `clean_status = complete` or `empty`. The tracker is displayed to confirm processing is finished before the regional cleanup step.

In [ ]:
process_stage_df = pd.read_pickle(g.Config.STAGE1_PROCESS_PATH)
process_stage_df

Reload the completed process tracker from disk. At this point all files should have `clean_status = complete` or `empty`. The tracker is displayed to confirm processing is finished before the regional cleanup step.

In [ ]:
for _, row in process_stage_df.iterrows():

    rejoin_done = None
    rejoin = None
    missing = None

    input_df = pd.read_json(row["input_path"])

    id_region_df = pd.read_pickle(row["id_region_path"])

    print(f"{_}-{row["input_path"]}-----------------")
    print(f"INIT REGION: {id_region_df.shape[0]}")
    if os.path.exists(row["clean_path"]):
        clean_df = pd.read_pickle(row["clean_path"])
        rejoin = id_region_df.merge(clean_df, on=["id"], how="left")
        rejoin_done = rejoin.dropna(subset=["clean_title"])
        missing = rejoin[rejoin["clean_title"].isnull()]
    else:
        rejoin = id_region_df
        missing = rejoin

    missing = missing.dropna(axis=1, how='all')

    if missing.shape[0] == 0:
        continue

    date = st1.extract_date_from_file_name(row["input_file"])

    for __, _row in process_stage_df[0:_].iterrows():

        if not os.path.exists(_row["clean_path"]):
            continue

        clean_df = pd.read_pickle(_row["clean_path"])

        rejoin = missing.merge(clean_df, on=["id"], how="left")
        #print(f"{_row["input_file"]} - rejoin: {rejoin.shape[0]}")

        if rejoin.shape[0] > 0:
            done = rejoin.dropna(subset=["clean_title"])

            missing = rejoin[rejoin["clean_title"].isnull()]
            if done.shape[0] > 0:
                rejoin_done = pd.concat([rejoin_done, done], ignore_index=True)
            if missing.shape[0] == 0:
                continue

            missing = missing.dropna(axis=1, how='all')

    key_cols = ["id", "region", "number_of_clicks", "date"]
    if missing.shape[0] > 0:
        id_region_df = id_region_df.merge(missing[key_cols], on=key_cols, how='left', indicator=True)
        id_region_df = id_region_df[id_region_df['_merge'] == 'left_only'].drop(columns=['_merge'])
        id_region_df.to_pickle(row["id_region_path"])
    print(f"FINAL REGION: {id_region_df.shape[0]}\n")


In [ ]:
pd.read_pickle(os.path.join(g.Config.STAGE1_OUTPUT_PATH, "ua-2024-01-01.pkl"))

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.